In [11]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Make the project root importable so we can `from gridsync import ...`
# regardless of whether Jupyter is launched from the repo root or data_pipeline/.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p for p in CANDIDATE_ROOTS if (p / "gridsync").is_dir()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
print(f"Project root: {PROJECT_ROOT}")

from gridsync import DenseEmbedder, SparseEmbedder, QdrantStore, hybrid_search, Neo4jStore

store = QdrantStore()
print(store.client.get_collections())

# dummy_text = "Stlol. Cat is a cute animal. It likes to play with mice. It might kill mice. It is a good hunter."
dummy_text = "Stpry of Dog. Dog is a cute animal. It likes to play with mice. It might kill mice. It is a good hunter."
dummy_metadata = {"source": "test_source", "author": "test_author"}



Project root: /Users/ayaanehsan/Developer/scsp/GridSync
collections=[CollectionDescription(name='electrical_grid_data')]


In [12]:
dense = DenseEmbedder()
sparse = SparseEmbedder()

dense_vec = dense.embed(dummy_text)
sparse_vec = sparse.embed(dummy_text)

store.ensure_hybrid_collection(dense_size=len(dense_vec))
store.upsert_hybrid(
    text=dummy_text,
    dense_vec=dense_vec,
    sparse_vec=sparse_vec,
    metadata=dummy_metadata,
)

print(store.info())

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=2 points_count=2 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors={'dense': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={'bm25': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5

In [13]:
results = hybrid_search(
    store=store,
    dense_embedder=dense,
    sparse_embedder=sparse,
    query_text="which animal play with mice?",
    top_k=3,
)
for r in results:
    print(f"Score: {r.score:.3f}  Dense: {r.dense_score}  Sparse: {r.sparse_score}  {r.payload}")

Score: 0.500  Dense: 0.0  Sparse: 1.0  {'text': 'Stlol. Cat is a cute animal. It likes to play with mice. It might kill mice. It is a good hunter.', 'source': 'test_source', 'author': 'test_author'}
Score: 0.500  Dense: 1.0  Sparse: 0.0  {'text': 'Stpry of Dog. Dog is a cute animal. It likes to play with mice. It might kill mice. It is a good hunter.', 'source': 'test_source', 'author': 'test_author'}


In [4]:
graph = Neo4jStore()
print("Neo4j connection successful")

Neo4j connection successful


In [6]:
graph.create_node("Event", {"name": "2023 Southwest Utah Disturbance", "date": "2023-04-10"})
graph.create_node("Location", {"name": "Southwest Utah", "state": "UT"})
graph.create_node("Utility", {"name": "WECC"})

graph.create_relationship(
    "Event", "2023 Southwest Utah Disturbance",
    "Location", "Southwest Utah",
    "OCCURRED_AT",
)
graph.create_relationship(
    "Utility", "WECC",
    "Event", "2023 Southwest Utah Disturbance",
    "REPORTED",
    properties={"source": "NERC/WECC Joint Staff Report"},
)

for record in graph.query():
    print(f"({record['from_node']}) -[{record['rel']}]-> ({record['to_node']})")

(2023 Southwest Utah Disturbance) -[OCCURRED_AT]-> (Southwest Utah)
(WECC) -[REPORTED]-> (2023 Southwest Utah Disturbance)


In [7]:
# Query relationships for a specific node
query = """
MATCH (a {name: $name})-[r]->(b)
RETURN a.name AS from_node, type(r) AS rel, b.name AS to_node
"""
graph.query(query, name="2023 Southwest Utah Disturbance")

[{'from_node': '2023 Southwest Utah Disturbance',
  'rel': 'OCCURRED_AT',
  'to_node': 'Southwest Utah'}]